> Trial new class

In [ ]:
import os
import torch
import numpy as np
import tempfile
import gdown
import pickle

SEED = 42

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")

from podcnf.roms import PODcnf
from podcnf.NFmodel import NormalizingFlow

from podcnf.DataGenerationStokes import *

Importing data and scalers

In [5]:
# Trained model
os.makedirs('../results/stokes', exist_ok=True)
target_folder = os.path.join("..", "results/stokes")
MODEL_NAME = os.path.join(target_folder, "NF_Stokes.pth")
gdown.download(id = "1o3TPXo87eKOHRo82LjsOxeVTWZ6Od3mQ", quiet=True, output = MODEL_NAME)
loaded_model = torch.load(MODEL_NAME, map_location = device)
loaded_params = loaded_model['hyperparameters']

# Data trasnformed through POD
os.makedirs('../data', exist_ok=True)
target_folder = os.path.join("..", "data") 
reduced_input_file = os.path.join(target_folder, "stokes_data_reduced_6400.pt")
gdown.download(id="19304ojlsmuL7hntN-m8KeR_CrAZD7wHb", quiet=True, output=reduced_input_file)
reduced_dataset = torch.load(reduced_input_file, weights_only=True)
mu = reduced_dataset['mu']
c = reduced_dataset['c']

# Raw data
os.makedirs('../data', exist_ok=True)
target_folder = os.path.join("..", "data")
filename = os.path.join(target_folder, "stokes_data_6400.pt")
gdown.download(id="1_E-gNMU9aMmWXHqm63JspI9i4kWy-wd1", quiet=True, output=filename)
loaded_data = torch.load(filename, weights_only=True)

u = loaded_data['u']
eps = loaded_data['eps'].squeeze(1)
theta = loaded_data['theta'].squeeze(1)
mu = loaded_data['mu']

dim_x = mu.shape[1]
dim_y = c.shape[1]
num_flows_loaded = loaded_params['num_flows']
hidden_size_loaded = loaded_params['hidden_size']
hidden_depth_loaded = loaded_params['hidden_depth']

print(f"NF parameters flows: {num_flows_loaded}, size: {hidden_size_loaded}, depth: {hidden_depth_loaded}")

NF_linear = NormalizingFlow(
    dim_x,
    dim_y,
    num_flows=num_flows_loaded,
    hidden_size=hidden_size_loaded,
    hidden_depth=hidden_depth_loaded,
    device=device
).to(device)

NF_linear.load_state_dict(loaded_model['model_state_dict'])

# c_scaler and mu_scaler
c_scaler_path = os.path.join(target_folder, "c_scaler.pkl")
if not os.path.exists(c_scaler_path):
    gdown.download(id="1m2RkT6bkPidlOOf9oB6EUgT4cYsCj9kd", quiet=True, output=c_scaler_path)
with open(c_scaler_path, "rb") as file:
    c_scaler = pickle.load(file)

mu_scaler_path = os.path.join(target_folder, "mu_scaler.pkl")
if not os.path.exists(mu_scaler_path):
    gdown.download(id="1USRvaurzv0nSxT3vHQ98AaapTE498Uj6", quiet=True, output=mu_scaler_path)
with open(mu_scaler_path, "rb") as file:
    mu_scaler = pickle.load(file)

from podcnf.roms import TorchScaler
mu_scaler = TorchScaler(mu_scaler.mean_, mu_scaler.scale_, device)
c_scaler =  TorchScaler(c_scaler.mean_, c_scaler.scale_, device)

# V_POD matrix
V_file = os.path.join(target_folder, "V_POD_matrix.pt")
if not os.path.exists(V_file):
    gdown.download(id="1dQ1QtqT8S96mJVZ-9axsPh5rW6tLO8RU", quiet=True, output=V_file)
V = torch.load(V_file, map_location="cpu", weights_only=False)

NF parameters flows: 24, size: 128, depth: 2


Defying the generator

In [7]:
podcnf = PODcnf(V, NF_linear, mu_scaler, c_scaler)

test_index = 6179 # np.random.randint(n_val, n_samples)
# Corresponding value for mu and g
mu_sele = mu[test_index - 1, :]
u_true = u[test_index -1, :]
print(f"mu_selected: {mu_sele}")
print(test_index)

u_sample = podcnf.sample(mu_sele, 2)
c_sample = podcnf.sample_latent(mu_sele, 2)

print(u_sample.shape)
print(c_sample.shape)

mu_selected: tensor([-1.1048,  4.0690, -2.5303])
6179
torch.Size([2, 2763])
torch.Size([2, 20])


In [ ]:
def plot_conditional_same_mu_stokes(n_generations,
                                    u_true, u_rec,
                                    Vh,
                                    mu_values=None):

    plt.figure(figsize=(9, 6))

    u_func_true = fe.Function(Vh)
    if isinstance(u_true, torch.Tensor):
        u_func_true.vector()[:] = u_true.detach().cpu().numpy()
    else:
        u_func_true.vector()[:] = u_true

    # True solution
    c = fe.plot(u_func_true, cmap='jet')
    plt.colorbar(c, shrink = 0.8, label='Concentration (u)')

    title_str = "True solution"
    if mu_values is not None:
        if isinstance(mu_values, torch.Tensor):
            mv = mu_values.detach().cpu().numpy().flatten()
        else:
            mv = np.array(mu_values).flatten()
        title_str += f"\n($c_1$={mv[0]:.2f}, $c_2$={mv[1]:.2f}, $c_3$={mv[2]:.2f})"

    plt.title(title_str)
    plt.show()

    print(f"{n_generations} generated samples:")

    cols = int(math.ceil(math.sqrt(n_generations)))
    rows = int(math.ceil(n_generations / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 5))
    if n_generations > 1:
        axes = axes.flatten()
    else:
        axes = [axes]

    # Generate sample
    for i in range(n_generations):
        ax = axes[i]
        plt.sca(ax)

        u_func_rec = fe.Function(Vh)

        sample_data = u_rec[i]
        if isinstance(sample_data, torch.Tensor):
            sample_data = sample_data.detach().cpu().numpy()

        u_func_rec.vector()[:] = sample_data

        c = fe.plot(u_func_rec, cmap='jet')
        plt.colorbar(c, shrink = 0.65, label='Concentration (u)')

        ax.set_title(f"Sample {i+1}")
        ax.set_xticks([])
        ax.set_yticks([])

    for i in range(n_generations, len(axes)):
        axes[i].axis('off')

    mu_str = f"\n($c_1$={mv[0]:.2f}, $c_2$={mv[1]:.2f}, $c_3$={mv[2]:.2f})"
    fig.suptitle(f"Conditional Generated Samples (Same $\\mu$) \n{mu_str}", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

In [ ]:
plot_conditional_same_mu_stokes(n_generations,
                                    u_true, u_sample,
                                    Vh,
                                    mu_values=None)